# Benchmarks

(benchmarks-overview)=

These studies compare fitted models with known generating functions or with
other libraries. The earlier comparison notebooks linked below remain under
`docs/book/cases/`; the later studies are in `docs/book/benchmarks/`.

- [](../cases/library_comparison.ipynb) and [](../cases/library_comparison_interactions.ipynb)
- [](../cases/distributional_comparison.ipynb)
- [](../cases/benchmark_insight.ipynb), [](../cases/benchmark_causal.ipynb), [](../cases/benchmark_runtime_api.ipynb)
- [](../cases/benchmark_structure.ipynb), [](../cases/benchmark_uncertainty.ipynb), [](../cases/benchmark_survival.ipynb)
- [](../cases/benchmark_classes.ipynb), [](../cases/benchmark_spatial.ipynb), [](../cases/benchmark_drivers.ipynb)
- [](../cases/benchmark_shapes.ipynb), [](../cases/benchmark_pairs.ipynb), [](../cases/benchmark_realdata.ipynb)
- [](../cases/benchmark_lss_drivers.ipynb) and [](../cases/benchmark_preexploration.ipynb)

- [](pair_form_stopping.ipynb): paired before/after stopping replay, competitor gaps and the scorecard profile.


(where-things-stand)=
## Benchmark status

For a summary of the results to date, the weaknesses found, and the work under way, see [](where_we_stand.ipynb).

## The knowledge-discovery scorecard

Every comparison in this part is graded against one instrument: the Knowledge
Discovery Scorecard, identifier `mboostrs-scorecard/2.1`. The canonical card is
`docs/scorecard/scorecard.md`.

The card grades recovery of true structure, reporting, uncertainty and prediction.
In fully automated work an AI agent conducts the analysis; in semi-automated
work a human analyst does.

A *data-generating process* (DGP) is the recipe
that produced the data, including which variables truly matter and how. A
*false discovery rate* (FDR) is the share of claimed findings that are wrong;
its partner, the true positive rate (TPR), is the share of real effects that
were found. The *information contract* specifies what the analyst receives before the
analysis, such as a family, formula or candidate predictors. Recovery scores
count only structure the analyst was asked to find and was not already given.

The card has twelve dimensions: D1a, D1b and D2 through D11. They are fixed
across studies, and the measurement battery is fixed before any run. Each is
scored 1 to 5 against an anchored rubric, where 1 is the weakest behaviour and
5 the strongest.

(information-contracts-511)=
### Analyst inputs (#511)

The old D1 asked a single question about relationship discovery, but discovery
depends on how much the analyst already knows. Version 2.1 splits it into D1a
and D1b under the information contracts fixed in
[#511](https://github.com/fcela/mboostrs/issues/511).

Under **Track A**, the analyst is handed a rough formula: the family and link,
the candidate predictors and the kinds of term to consider. The true structure
is not supplied. Only what the formula left *open* is scored; a term the formula
already named earns no discovery credit. Under **Track B**, the analyst gets the
data, the meaning of the response, the metadata, the design and a budget, but no
formula at all. Every piece of recovered structure is scored.

A middle track, **partial knowledge**, supplies some components and leaves others
for the analyst to find. Only the latter count toward recovery; the protocol
must list the supplied components.

The comparison reference depends on whether the generating model is known. On planted-truth data the
generator is an oracle, so a library's loss can be measured against the true
structure. On real data there is no oracle, so the comparison is a
*prespecified* reference specification, never a best fit chosen after seeing the
test loss. Tuning and screening are predictive decisions, not calibrated
hypothesis tests. Where losses are reported, the card prefers paired per-task
differences, an excess over the reference and a gap to the oracle, over pooled
averages, and asks for uncertainty across tasks and seeds.


### D1a. Relationship recovery under a prescribed specification

**What it covers.** D1a measures whether a library finds the structure that a
rough formula left open. The analyst supplies the family, the candidate
predictors and the term types, but not the true shape.

**Why it matters.** Real analyses rarely start from nothing. A statistician
brings domain knowledge, writes a tentative model, then asks what it missed. A library that fits only the named terms leaves the analyst to search for omitted
structure. D1a rewards a library that searches the open part and controls its
false discoveries while doing so.

**How it is measured.** Freeze the prescribed formula. Plant truth that adds
structure the formula omits: interactions, nonlinearities, heteroscedasticity
and pure-noise variables. Report the true positive rate and false discovery
rate per structure type for the open terms only, and list the prescribed terms
so no credit leaks to them. The rubric runs from a literal fit with no search
(1), through ad-hoc importance scanning with no error control (2), to a
structured search with stated assumptions (3), to planted-truth validation with
the false-discovery behaviour characterized (4), and finally to higher-order
open structure with published recovery rates (5).

**Illustration.** Prescribe a smooth additive model for wages. Plant a true
education-by-experience interaction that the formula never names. A level-1
library returns the smooth terms and stops; a level-4 library flags the
interaction and reports how often its open-structure claims are wrong.


### D1b. Relationship discovery from data alone

**What it covers.** D1b measures finding true structure with no formula at all.

**Why it matters.** Without a supplied formula, the analyst must identify candidate effects as well
as estimate them. Scoring false discoveries alone would reward a method that
always reports nothing, so D1b also measures recovery of planted structure.

**How it is measured.** Supply the data, response semantics, metadata, design
and budget, but no formula. Score the true positive rate and false discovery
rate by structure type against planted truth. A library that reports nothing cannot outscore one that reports true structure;
on a pure-noise task it must report nothing. The rubric runs from manual
specification only (1), through ad-hoc aids with no error control (2), to
structured screening with stated assumptions (3), to planted-truth validation
with characterized false discovery (4), and finally to higher-order structure
with published recovery rates (5). The protocol must publish the discovery
budget and the search scope.

**Illustration.** On a table with no signal, the target is no reported effects. A level-3 library returns a small screened set with stated
rules and reports none of it when the noise test comes up empty; a level-2
importance plot always highlights some variable and so manufactures discovery.


### D2. Causal humility and support

**What it covers.** D2 measures whether a library stops an analyst from reading
causes out of a predictive fit, and supports causal estimation with stated identification assumptions.

**Why it matters.** A model can predict well while its top drivers are
confounded. Treating a predictive ranking as a cause leads to bad decisions,
especially when the analyst is told to trust the top of the list.

**How it is measured.** A documentation audit plus a workflow checklist:
whether findings are scoped as associational, whether confounder adjustment is
supported, and whether sensitivity checks are available. The rubric runs from
language that encourages a causal reading (1), through neutral (2), to
associational scope with adjustment support (3), to built-in sensitivity checks
for key findings (4), and finally to experimental-design support integrated with
modelling (5).

**Worked example.** A fit reports that a job-training program drives earnings.
The data also show that participants were older on average, and age raises
earnings. A predictive-only library prints training at the top of the driver
list with no caveat: that is the level-1 behaviour. A level-4 workflow identifies the association as observational, adjusts for age
and prior earnings, and checks sensitivity to an unmeasured confounder. In this
example it reports a smaller effect with an uncertainty interval and states the
assumptions needed for a causal interpretation.

**Nearby dimensions.** D2 does not grade whether structure was found (D1a and
D1b) or whether predictions generalise (D11). Correct discovery followed by a
causal misreading is a D2 failure alone.


### D3. Explanation quality

**What it covers.** D3 measures how well a library turns findings into a
faithful, complete and decision-useful narrative.

**Why it matters.** Analysts need to explain which effects the model supports and which limitations
affect a decision. D3 grades the report produced from the fitted model.

**How it is measured.** Analysts or agents write reports from library outputs
on a planted-truth dataset. Graders compare the reports with the key on two
axes: faithfulness, meaning the story matches the fitted model and the truth,
and completeness, meaning the known effects and the limitations are covered.
The protocol records whether the run was Track A or Track B and separates
explanation credit for discovered and prescribed terms. The rubric runs from no
explanation outputs (1), through per-prediction summaries only (2), to faithful
dataset-level summaries such as effect plots and importance rankings (3), to
ranked, evidence-graded findings with uncertainty that can be used in a report (4), and finally to a complete
account of supported findings, unresolved questions and further checks (5).

**Contrast with D4.** D3 owns the narrative and its faithfulness; D4 owns the
effect size and its uncertainty. Two libraries can see the same wage regression
and diverge. Library A returns a fluent, well-ordered summary of which
predictors matter, but every number is a point value with no interval: this can score high on D3 and low on D4. Library B returns a
raw coefficient table where each effect carries a confidence interval that
covers the truth about as often as it claims, but there is no story, no ranking
and no plain-language summary: high D4 and low D3.


### D4. Importance quantification

**What it covers.** D4 measures whether a library reports effect sizes with
uncertainty intervals whose coverage can be assessed, rather than only rankings
or heuristic scores.

**Why it matters.** An importance ranking orders predictors but gives neither an effect magnitude
nor its uncertainty. Saying that a variable is important is not the same as
saying how much the outcome moves when it changes.

**How it is measured.** Coverage calibration on planted-truth DGPs, plus the
interval width needed to reach a given coverage. A well-behaved 90% interval
should contain the true effect about 90% of the time; a library that reaches
that coverage only by making intervals enormous is not credited. Under Track A,
intervals for discovered (open) effects are reported separately from prescribed
effects. The rubric runs from point estimates only (1), through heuristic
scores with no sampling interpretation (2), to intervals under stated
assumptions (3), to empirically validated coverage (4), and finally to
decision-relevant magnitudes on the outcome scale (5).

**Illustration.** One library reports that education raises wages by 1.2 log
points with a 90% interval of 0.9 to 1.5, and those intervals contain the truth
90% of the time; that is level 4. Another reports that education is important,
with a gain score of 0.42 and no scale or error; that is level 2.

**Nearby dimensions.** D4 and D9 both concern intervals but not the same ones.
D4 covers effect intervals: a coefficient, an average treatment effect, an
effect curve. D9 covers prediction intervals: a new observation. A library can
calibrate one and not the other. D4 and D11 also differ: D4 asks whether the
effect estimate and its interval are accurate; D11 asks whether predictions
generalise.


### D5. Experimental iteration speed

**What it covers.** D5 measures the wall-clock time from the question, what if
we tried X, to having an answer. D5 always means iteration speed.

**Why it matters.** Fit and diagnostic times determine how many alternative specifications an
analyst can evaluate in a session.

**How it is measured.** A standardized analyst session performs ten specified
follow-up fits on 50k-by-20 data, including diagnostics, and reports the median
and the 90th-percentile wall-clock time. The rubric runs from fits that take
minutes and make iteration impractical (1), through tens of seconds (2), to
seconds per fit with cheap diagnostics (3), to sub-second refits and batched
counterfactuals (4), and finally to incremental or online updating where a
follow-up does not need a full refit (5).

**Illustration.** Consider ten refits, each changing a spline knot and inspecting the effect.
At forty seconds per refit, the fits take almost seven minutes; at four tenths
of a second, they take a few seconds.

**Nearby dimension.** D5 is raw latency. D8 is whether an autonomous agent can
drive the loop at all through the API. A fast library with untyped,
unparseable outputs can be quick and still agent-hostile, and a slower library
with documented inputs, outputs and errors can support autonomous analysis.


### D6. Robustness across conditions

**What it covers.** D6 measures graceful behaviour on small samples, large
samples, messy data and shifted data.

**Why it matters.** Real tables arrive with missing values, outliers and mixed
types, and they grow or shrink. Requiring complete numeric inputs leaves the analyst to handle missing values
and categorical predictors before fitting.

**How it is measured.** A fixed battery: a small-n case (n = 200), a messy case
(15% missing completely at random, plus outliers and mixed types), and a
large-n case (five million rows). The protocol reports D1a and D1b recovery and
runtime per condition, under both information contracts where that is feasible.
The rubric runs from clean complete numeric matrices only (1), through basic
missing and mixed-type handling (2), to measured recovery across small-n and
messy conditions with documented degradation (3), to all of that plus large-n
scalability (4), and finally to shift and stress diagnostics that warn when
findings may not transfer (5).

**Illustration.** Delete 15% of the cells at random and add 1% outliers. A
level-3 library keeps most of its recovery and records the drop; a level-1
library errors on the first missing cell.


### D7. Human-friendliness

**What it covers.** D7 measures learnability, diagnosability and trust for a
competent human analyst.

**Why it matters.** The analyst is part of the discovery loop. Cryptic errors,
misleading defaults and unguided workflows increase the time needed to complete and diagnose an analysis.

**How it is measured.** A task-based study measuring time-to-first-discovery and
the rate at which an error is recovered from, with issue-tracker mining as a
fallback proxy. It is run under both Track A and Track B, because the guidance a
user needs differs when a formula is prescribed. The rubric runs from cryptic
APIs and misleading defaults (1), through usable-with-effort (2), to a coherent
API with guided workflows and informative errors (3), to executable case studies
for the main patterns and diagnostics that point at fixes (4), and finally to
direct evidence that the documentation works (5).

**Illustration.** A mistyped family raises a message that names the problem,
suggests the intended family and links its page, which is level 3, against a
stack trace that ends somewhere inside a compiled backend, which is level 1.


### D8. Agent-friendliness

**What it covers.** D8 measures whether an autonomous agent can drive the whole
discovery loop through the API.

**Why it matters.** In the fully automated setting the analyst is a program. Structured findings and errors that name a remedy let the agent choose its next
API call without reconstructing those details from a plot or prose.

**How it is measured.** A frozen-prompt language-model agent performs discovery
on planted-truth datasets using only the library. The protocol grades D1a and
D1b recovery and D3 report quality, and runs Track A and Track B separately,
naming the prescribed terms for Track A. The rubric runs from every step needing
human judgment (1), through a programmatic API with unstructured outputs (2), to
structured outputs such as typed findings and machine-readable diagnostics with
actionable error codes (3), to a documented agent contract with assumptions,
checks and validation hooks (4), and finally to a demonstrated end-to-end
completion with no human assistance (5).

**Illustration.** A diagnostic call returns a typed table of findings with a
severity and a suggested next call, versus a call that returns a free-text
paragraph the agent must parse and guess at.


### D9. Prediction-interval calibration

**What it covers.** D9 measures whether stated uncertainty matches reality and
widens when the data warrant. D9 always means calibration.

**Why it matters.** An interval advertised at 90% coverage can lead an analyst to underestimate
prediction error if it covers substantially fewer than 90% of new outcomes.

**How it is measured.** Coverage experiments on planted-truth DGPs, including
heteroscedastic and small-n variants, comparing empirical coverage with nominal
coverage. Results are attributed to the track that produced the estimate, so
calibration is not credited to structure that was supplied. The rubric runs from
no uncertainty quantification (1), through uncertainty that is reported but
uncalibrated (2), to approximately nominal coverage on standard DGPs (3), to
adaptive bands that widen under heteroscedasticity, small n or extrapolation
(4), and finally to diagnostics that warn when the uncertainty itself is
unreliable (5).

**Illustration.** Under a heteroscedastic DGP, a level-3 library's bands are
nominal on average but flat, so they under-cover in the high-variance region; a
level-4 library lets the band widen there.


### D10. Reproducibility and auditability

**What it covers.** D10 measures whether a third party can reproduce every
number and audit every decision.

**Why it matters.** Reproducing a result requires the data, settings, seeds and code that produced
it. Records of rejected candidates are also needed to inspect model selection.

**How it is measured.** An independent party reproduces the report from the
report alone, and graders score numerical agreement and the completeness of the
decision trail. The rubric runs from undocumented randomness or hidden state
(1), through fixable seeds with no decision trail (2), to a full decision log of
candidates, adoptions and rejections with reasons and seeds (3), to one-command
reproduction of the whole report (4), and finally to artifact versioning that
pins data, code and environment hashes alongside the findings (5).

**Illustration.** A second analyst reruns from the report, recovers the same
effects to the last digit, and can see from the log which candidates were
considered and why each was dropped; that is level 4, against a script that
prints a number while hiding its seed and its search.

**Nearby dimension.** D3 is whether the story is faithful for a reader; D10 is
whether an independent party can reproduce the numbers. A persuasive report with
no seed or decision trail can be high D3 and low D10.


### D11. Predictive validity

**What it covers.** D11 measures useful predictions on observations that were
not used to construct or select the model.

**Why it matters.** Leakage and model selection on test responses can lower the reported loss
without improving prediction on new observations. D11 therefore requires data
excluded from preprocessing, fitting and selection.

**How it is measured.** Freeze the estimand, the information available at
prediction time, the loss and the reference population. Use untouched outer test
data or nested validation, and keep every learned preprocessing step, screen and
tuning choice inside the training data. Report the per-task loss, paired
differences with uncertainty, the comparators, the sample sizes, the failures and
the test-use history. On real data the prespecified reference is not an oracle,
so paired loss differences compare the fitted model with that reference rather
than with known optimal predictions.
Classification, mean prediction, distribution prediction and causal-effect
prediction are reported as separate targets and are not pooled indiscriminately.
The rubric runs from measured leakage, invalid prediction timing or failure to
beat a simple baseline (1), through limited evidence (2), to outer
validation that beats a prespecified baseline with uncertainty and failures
reported (3), to competitive performance across task families under matched
budgets (4), and finally to confirmation on held-back families and deployment
splits, with transfer limits characterized (5).

**Contrast with D1a and D1b.** D1a and D1b grade recovery of structure under the specified analyst inputs; D11 grades prediction on outer-test rows. A spline model can recover a planted interaction
with a controlled false-discovery rate and still lose on outer log-loss: high
D1b, low D11. A gradient-boosted black box can top the held-out log-loss while
reporting nothing about which effects it learned: high D11, low D1b.


(what-goes-where)=
## Metric assignments

Most reported numbers belong to exactly one dimension. This table maps the
common ones, and names the dimensions they are not.

| Claim or artifact | Scored under | Excluded from |
|---|---|---|
| Held-out RMSE or log-loss on untouched data | D11 | Not D1a or D1b: a good loss says nothing about which structure was recovered. |
| Recovered planted interaction with a controlled false-discovery rate | D1a if a prescribed formula left it open, otherwise D1b | Not D11: recovery is not prediction. |
| Fluent per-feature story with no intervals | D3 | Not D4: narrative is not magnitude or coverage. |
| Effect size with a confidence interval | D4 | Not D9: this is an effect interval, not a prediction interval. |
| Prediction-band coverage over new observations | D9 | Not D4: prediction calibration, not effect calibration. |
| Average treatment effect with a sensitivity analysis | D2 for the causal scope, D4 for the magnitude and interval | Not D1a or D1b: a claim about a designated effect, not structural recovery. |
| Time for ten refits on 50k-by-20 data | D5 | Not D8: latency is not agent autonomy. |
| An agent runs the loop from a frozen prompt with no assists | D8 | Not D7: autonomy is not human learnability. |
| A second analyst reproduces every number from the report | D10 | Not D3: reproduction is not a readable narrative. |
| Recovery holds on n = 200 and on messy data | D6 | Not D1a or D1b: this is how recovery behaves across conditions. |
| Reports nothing on a pure-noise table | D1b | Not D1a: silence is a data-alone discovery property. |


(compact-cheat-sheet)=
## Scorecard dimensions

| Dimension | Measures | Excludes |
|---|---|---|
| D1a | Recovery of structure a prescribed formula left open | Terms the formula named; prediction |
| D1b | Discovery from data alone, including no claims on noise | Prescribed-specification credit; prediction |
| D2 | Causal humility and support | Structure recovery; prediction |
| D3 | Narrative faithfulness and completeness | Effect magnitude and coverage |
| D4 | Effect sizes and their interval coverage | Prediction intervals; held-out loss |
| D5 | Iteration latency | Agent autonomy |
| D6 | Robustness across small, large, messy and shifted data | Baseline discovery accuracy |
| D7 | Human learnability and diagnosability | Autonomous agent behaviour |
| D8 | Autonomous agent completion through the API | Human-friendliness |
| D9 | Prediction-interval calibration | Effect intervals |
| D10 | Reproduction and model-selection records | Report readability |
| D11 | Generalisation to outer data | Which structure was recovered |

An aggregate is optional and subordinate. The full profile is the primary
report. A weighted aggregate is permitted only when all twelve dimensions are
scored on one prespecified battery in one operating mode. The mode weights are:

| Dimension | Fully automated | Semi-automated |
|---|---|---|
| D1a Relationship recovery (prescribed) | 2× | 2× |
| D1b Relationship discovery (data alone) | 2× | 2× |
| D2 Causal humility | 1× | 1× |
| D3 Explanation / story | 2× | 1× |
| D4 Importance quantification | 1× | 2× |
| D5 Iteration speed | 2× | 1× |
| D6 Robustness | 1× | 1× |
| D7 Human-friendliness | 0.5× | 2× |
| D8 Agent-friendliness | 2× | 0.5× |
| D9 Uncertainty honesty | 2× | 1× |
| D10 Reproducibility | 1× | 1× |
| D11 Predictive validity | 2× | 2× |

No score is averaged across studies or renormalized from an incomplete profile.
A dimension without evidence is `unmeasured`, not a 1 or a 0. The full anchor
tables, the required measurements and comparison rules are in
`docs/scorecard/scorecard.md`; `docs/scorecard/README.md` is the index, and
`docs/scorecard/status.md` records the present evidence status.

Two pages in this part apply the card directly. [](claims_discovery.ipynb) and
[](causal_discovery.ipynb) freeze their protocols against the canonical anchors
and report the per-dimension profile for every library, with the weights above.
Nothing is re-scored here; the public-data pages compare held-out scores and
resources without grading against the card.


## Public data sets against other libraries

Five pages score mboostrs against statsmodels, pyGAM, scikit-learn, LightGBM,
XGBoost and CatBoost on public data sets with a published analysis, one
modelling problem each. Every method sees the same random splits. The pages
report held-out scores with their split-to-split uncertainty, the time and peak
memory of a fit, the warnings and failures each method raised, and the code each
one took. The accompanying analyses relate the score differences to the measured response,
predictors and model specifications.

- [](public_count.ipynb) — over-dispersed counts (Long's bioChemists): what the family buys against what the mean model buys.
- [](public_binary.ipynb) — labour-force participation (Mroz): an imputed covariate that rewards flexibility for the wrong reason.
- [](public_spatial.ipynb) — heavy metals on the Meuse floodplain: covariates, a spatial surface, kriging and trees on 155 points.
- [](public_distributional.ipynb) — wages and their spread (CPS 1988): mean-only models against models of the variance.
- [](public_scaling.ipynb) — fit time and peak memory as the number of rows grows, at a fixed amount of boosting.

All five read artifacts written by `benches/bench_public_tasks.py`. The artifacts
pin the data digests, seeds, thread counts and library versions.


## Agent analysts on a planted-truth problem

[](claims_discovery.ipynb) gives seven libraries (mboostrs, XGBoost, LightGBM,
CatBoost, InterpretML EBM, pyGAM, statsmodels) to seven Claude Haiku analysts,
one each, with the same written brief. The data are the real California census
covariates with a simulated claim-cost outcome, so every discovery claim, effect
interval and prediction interval is scored against the truth. Each analyst's
pipeline is re-run, unseen, on twelve hidden draws, including two with no signal
and two with 800 rows, and the results are graded on the knowledge-discovery
scorecard.

[](causal_discovery.ipynb) gives five libraries (mboostrs, EconML, DoubleML,
statsmodels, LightGBM) to five Claude Haiku analysts with a written brief for a
program evaluation: a non-randomized job-training program, follow-up earnings
missing for 16% of applicants, and thin overlap above an eligibility limit. The
applicants are real 1988 survey records; enrolment and outcomes are simulated
from a known model, so average effects, per-applicant effects, causal roles and
claims on data with no effect are scored against the truth. The mboostrs column
is the fourth mboostrs analyst, after three rounds of library fixes;
`causal_discovery_before.ipynb` in the same directory records the earlier runs.


## Predictive validity in two tracks

[](predictive_tracks.ipynb) compares supplied-structure fitting and data-alone
discovery with four validation-tuned tree libraries on independent smooth,
interaction and factor-curve draws. It reports paired gaps and SEs, all methods
and seeds, the reference-level dictionary defect, separate schema and candidate
repairs, fresh controls and an unchanged full rerun. Only D11 is graded;
the other eleven dimensions remain unmeasured, with no aggregate. Executed
outputs are stored under the current benchmark guidance.


## Predictor encoding

[](encoding_tracks.ipynb) uses five encodings of the same eight latent draws
in both information tracks, comparing four tuned tree libraries and Ridge.
It reports the binary-reference repair, a small signed regression, the two
remaining shifted-budget gaps and separate stopping-range/maximum-grid
diagnostics. Eight of ten original-budget cells meet the 1% target. Only D11
is graded; other dimensions stay unmeasured. Executed outputs are retained.


## D5 gap-closing round 3

[](d5_gap_closing_round3.ipynb) examines four gaps from issue #602: the quine Poisson screening-rate bug, large-n quantile binning (3.3x speedup, faster than LightGBM in the recorded comparison), cancer probability scoring, and joint-boundary representation. The corrected cancer scoring does not establish a Binomial engine defect; later boundary studies test pair representation and main-effect selection.